时光旅行
***
- 重放
- 分叉

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import END, StateGraph, MessageState, START
from langgraph.checkpoint.memory import MemorySaver

@tool
def play_song_on_qq(song: str):
    """在qq音乐上播放歌曲"""
    # 调用QQ音乐API
    return f"成功在QQ音乐上播放{song}!"

@tool
def play_song_on_163(song: str):
    """在163音乐上播放歌曲"""
    # 调用163音乐API
    return f"成功在163音乐上播放{song}!"


tools = [play_song_on_qq, play_song_on_163]
tool_node = ToolNode(tools)


from langchain_openai import ChatOpenAI
import os 

deepseek = ChatOpenAI(
   model="gpt-4o",
   temperature=0,
   api_key=os.getenv("DEEPSEEK_API_KEY"),
   base_url="https://api.deepseek.cn/v1",
)

model = deepseek.bind_tools(bools, parallel_tool_calls=False)

# 定义确定是否继续的函数
def should_continue(state):
    messages = state["messages"]
    last_message = messages[-1]
    # 如果没有函数调用，则结束
    if not last_message.tool_calls:
        return "end"
    # 否则如果有，我们继续
    else:
        return "continue"

#  定义调用模型的函数
def call_model(state):
    messages = state["messages"]
    response = model.invoke(messages)
    # 我们返回呈个列表，因为这将被添加到现有列表中
    return {"messages": [response]}

#  定义一个新图
workflow = StateGraph(MessageState)

# 定义我们将循环的两个节点
workflow.add_node("agent", call_model)
workflow.add_node("action", tool_node)

# 设置入口
workflow.add_edge(START, "agent")

# 现在添加一个条件边
workflow.add_conditional_edge(
    "agent", 
    should_continue, 
    {
        "continue": "action",
        "end": END,
    }
)

workflow.add_edge("action", "agent")

# 设置内存
memory = MemorySaver()

app = workflow.compile(checkpointer=memory)

进行简单的交互，要求插入周杰伦的歌曲

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id" : "1"}}
input_message = HumanMessage(content="你能播放一首周杰伦播放量最高的歌曲吗？")
for event in app.stream({"messages": [input_message]}, config, stream_mode="values"):
    event['messages'][-1].pretty_print()

查看记录并重放

In [ ]:
app.get_state(config).values["messages"]

In [ ]:
all_states = []
for state in app.get_state_history(config):
    print(state)
    all_states.append(state)
    print('--')

我们可以返回任一状态节点，并从那个时候重新开始操作

In [ ]:
to_replay = all_states[2]
to_replay.values
to_replay.next

如果相从这个状态节点重播，只需要这样

In [ ]:
for event in app.stream(None, to_replay.config):
    for v in event.values():
        print(v)

分叉操作
*** 
从某个节点开始操作，对执行数据进行分叉

In [ ]:
# 修改最后一个消息的工具调用
# 我们将其从play_song_on_qq更为play_song_on_qq_163
last_message = to_replay.values["messages"][-1]
last_message.tool_calls[0]["name"] = "play_song_on_163"

branch_config = app.update_state(
    to_replay.config,
    {"messages": [last_message]}
)